# Task 1: traders having an unusually bad session
Same analysis as `01_exploration.ipynb`, with the logic in `src/arrowfin/`. This notebook only calls it and produces the ranked ten.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from arrowfin import load, sessions, pnl, quality, ranking
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

## 1. Load, CME sessions and P&L per (account, session)

In [2]:
t = load.load_all(ROOT / "dataset")                       # schema asserts
fills = sessions.add_session(t["fills"])                   # CME session (trade date)
cal = sessions.session_calendar(fills)
fp = pnl.add_realized_pnl(fills, t["instruments"])         # average cost + point value
op = pnl.open_positions(fp, t["market_prices"])            # unrealized at the cut
sess = pnl.session_pnl(fp, op, t["accounts"], t["traders"])
sess[["realized_gross_usd", "commissions_usd", "unrealized_usd", "net_pnl_usd"]].sum().round(0)

realized_gross_usd   -43209.0
commissions_usd       22099.0
unrealized_usd        -2044.0
net_pnl_usd          -67352.0
dtype: float64

## 2. Data quality findings

In [3]:
dq = quality.quality_report(t, fp, cal, op, sess)
dq[["check", "severity", "entity", "detail"]]

,check,severity,entity,detail
0,partial_session,high,2026-08-25,Session 2026-08-25 is 16.5 of 23 h old at the ...
1,inactive_account_with_fills,high,ACC-2053 (T-040),"status=closed, 20 fills between 2026-08-10 and..."
2,inactive_account_with_fills,high,ACC-2054 (T-041),"status=restricted, 3 fills between 2026-08-24 ..."
3,zero_balance_with_fills,high,ACC-2028 (T-025),"balance=0.0, buying_power=0.0, yet it trades."
4,short_history,high,ACC-2027,Only 1 session(s) with fills.
5,short_history,high,ACC-2028,Only 1 session(s) with fills.
6,short_history,high,ACC-2029,Only 2 session(s) with fills.
7,short_history,high,ACC-2054,Only 2 session(s) with fills.
8,exceeds_max_position,high,ACC-2013,Max position 9 contracts vs limit 5.
9,exceeds_max_position,high,ACC-2025,Max position 32 contracts vs limit 20.


## 3. Ranking: today's session against each trader's own normal

In [4]:
tb = ranking.add_baseline(ranking.trader_sessions(sess))
today = ranking.score_session(tb, cal["session_date"].max())
top10, no_history = ranking.top_n(today, 10)
top10[["rank", "trader_id", "n_accounts", "contracts", "base_contracts", "size_vs_normal",
       "net_pnl_usd", "base_pnl", "base_spread", "worst_prior", "worse_than_worst_prior", "z"]].round(2)

,rank,trader_id,n_accounts,contracts,base_contracts,size_vs_normal,net_pnl_usd,base_pnl,base_spread,worst_prior,worse_than_worst_prior,z
131,1,T-011,1,14,14.0,1.00,-758.55,15.70,59.23,-246.10,True,-13.07
212,2,T-018,3,24,28.0,0.86,-1516.80,-130.77,125.32,-350.85,True,-11.06
222,3,T-019,2,26,24.0,1.08,-1559.20,129.30,202.52,-494.50,True,-8.34
107,4,T-009,1,12,10.0,1.20,-669.95,-30.60,86.88,-252.30,True,-7.36
234,5,T-020,1,52,28.0,1.86,-2397.65,17.45,397.26,-1267.40,True,-6.08
143,6,T-012,1,18,14.0,1.29,-724.10,58.35,151.00,-320.20,True,-5.18
270,7,T-023,1,19,20.0,0.95,-2486.10,-344.00,429.95,-919.40,True,-4.98
472,8,T-044,1,4,14.0,0.29,-559.92,225.30,233.87,-1016.50,False,-3.36
95,9,T-008,1,10,12.0,0.83,-459.00,-35.50,157.75,-208.40,True,-2.68
307,10,T-029,1,12,15.0,0.80,-413.90,-72.60,130.39,-559.20,False,-2.62


In [5]:
no_history[["trader_id", "n_prior", "contracts", "net_pnl_usd"]]

,trader_id,n_prior,contracts,net_pnl_usd
271,T-024,0,38,-1579.60
272,T-025,0,5,-72.00
274,T-026,1,6,-127.95


In [6]:
out = ROOT / "outputs"; out.mkdir(exist_ok=True)
top10.to_csv(out / "top10.csv", index=False)
no_history.to_csv(out / "no_history.csv", index=False)
dq.to_csv(out / "data_quality.csv", index=False)
sess.to_csv(out / "account_sessions.csv", index=False)